# 010: GPT Autoregressive Modeling — Causal masking, autoregressive language modeling, and temperature sampling

## 1. AUTOREGRESSIVE LANGUAGE MODELING
- GPT models are autoregressive: they predict the next token based strictly on the sequence of preceding tokens:
  $$P(X) = \prod_{t=1}^{T} P(x_t | x_{<t})$$
- This is implemented via **causal masking** in the Self-Attention layer. A lower-triangular attention mask zeroes out attention scores for future tokens:
  $$\text{Mask}_{ij} = 0 \text{ if } i \geq j \text{ else } -\infty$$

## 2. TEMPERATURE SAMPLING
- The raw logits $z_i$ from the model are scaled by a **temperature** parameter $T > 0$ before applying softmax:
  $$p_i = \frac{e^{z_i / T}}{\sum_j e^{z_j / T}}$$
  - $T \to 0$: distribution becomes highly peaky (greedy selection, high confidence but repetitive).
  - $T \to 1$: standard softmax.
  - $T > 1$: flattens the distribution, increasing diversity/randomness but risking gibberish.
---


In [ ]:
import numpy as np

def softmax(x, axis=-1):
    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e_x / np.sum(e_x, axis=axis, keepdims=True)

class AutoregressiveLMGenerator:
    """Generates text autoregressively using causal predictions and temperature sampling."""
    def __init__(self, vocab):
        self.vocab = vocab
        self.vocab_size = len(vocab)
        # Mock weights for next-token logit prediction: maps context tokens to logits
        np.random.seed(42)
        self.context_weights = np.random.randn(self.vocab_size, self.vocab_size) * 1.5

    def predict_logits(self, context_indices):
        """Mock logit generator based on sum of context embeddings."""
        if not context_indices:
            return np.zeros(self.vocab_size)
        # Sum logit vectors for context tokens
        logits = np.sum(self.context_weights[context_indices, :], axis=0)
        return logits

    def sample_next_token(self, context_indices, temperature=1.0):
        """Samples the next token index given context and temperature."""
        logits = self.predict_logits(context_indices)
        
        # Apply temperature scaling
        if temperature <= 0.0:
            # Greedy decoding
            return int(np.argmax(logits))
            
        scaled_logits = logits / temperature
        probs = softmax(scaled_logits)
        
        # Sample from probability distribution
        return int(np.random.choice(self.vocab_size, p=probs))

    def generate(self, prompt, max_len=10, temperature=1.0):
        """Autoregressive generation loop."""
        generated = list(prompt)
        print(f"Prompt: {' '.join(prompt)}")
        
        for _ in range(max_len):
            context_indices = [self.vocab.index(w) for w in generated if w in self.vocab]
            next_idx = self.sample_next_token(context_indices, temperature)
            next_word = self.vocab[next_idx]
            generated.append(next_word)
            
            if next_word == "</s>": # End of sentence token
                break
        return generated



In [ ]:
vocab = ["the", "cat", "sat", "on", "mat", "dog", "ran", "chased", "squirrel", "</s>"]
generator = AutoregressiveLMGenerator(vocab)

print("--- Text Generation with Temperature Variations ---")
prompt = ["the", "cat"]

# Test greedy decoding (T = 0)
greedy_out = generator.generate(prompt, max_len=6, temperature=0.0)
print(f"Greedy Output (T=0):  {' '.join(greedy_out)}")

# Test standard sampling (T = 1.0)
np.random.seed(42)
normal_out = generator.generate(prompt, max_len=6, temperature=1.0)
print(f"Sampled Output (T=1.0): {' '.join(normal_out)}")

# Test high entropy sampling (T = 3.0)
high_out = generator.generate(prompt, max_len=6, temperature=3.0)
print(f"Sampled Output (T=3.0): {' '.join(high_out)}")
